In [1]:
import sys
sys.path.append("..")

import gc
from src.datos import cargar_obt, agregar_features
from src.pipeline import preparar_datos, configurar_mlflow, evaluar_y_registrar

df = agregar_features(cargar_obt())
entrenamiento, prueba = preparar_datos(df)
del df
gc.collect()

configurar_mlflow()
print("Entrenamiento:", entrenamiento.shape, "| Prueba:", prueba.shape)

Entrenamiento: (757018, 59) | Prueba: (40282, 59)


In [2]:
from sklearn.dummy import DummyRegressor

run_id, cv = evaluar_y_registrar(
    DummyRegressor(strategy="median"),
    nombre="baseline_ingenua_mediana",
    familia="dummy",
    fase="baseline",
    entrenamiento=entrenamiento,
)

print("WAPE CV:", round(-cv["test_wape"].mean(), 4), "±", round(cv["test_wape"].std(), 4))
print("Costo CV:", round(-cv["test_costo"].mean(), 4))

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run baseline_ingenua_mediana at: http://localhost:5000/#/experiments/1/runs/a89e5b08119a46939973b4ddd72284f9
🧪 View experiment at: http://localhost:5000/#/experiments/1
WAPE CV: 0.3123 ± 0.0127
Costo CV: 0.527


In [2]:
from src.pipeline import ReglaNegocio

run_id, cv = evaluar_y_registrar(
    ReglaNegocio(),
    nombre="baseline_negocio_regla_pset1",
    familia="regla",
    fase="baseline",
    entrenamiento=entrenamiento,
    preprocesar=False,
)

print("WAPE CV:", round(-cv["test_wape"].mean(), 4), "±", round(cv["test_wape"].std(), 4))
print("Costo CV:", round(-cv["test_costo"].mean(), 4))

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run baseline_negocio_regla_pset1 at: http://localhost:5000/#/experiments/1/runs/2ef70d702068433ab8461de0a8a8a950
🧪 View experiment at: http://localhost:5000/#/experiments/1
WAPE CV: 0.1746 ± 0.024
Costo CV: 0.2709


In [3]:
from sklearn.linear_model import Ridge

def resumen(cv):
    tr = -cv["train_wape"].mean()
    va = -cv["test_wape"].mean()
    print(f"WAPE train: {tr:.4f} | WAPE CV: {va:.4f} ± {cv['test_wape'].std():.4f} | brecha: {va - tr:.4f}")
    print(f"Costo CV: {-cv['test_costo'].mean():.4f}")

run_id, cv = evaluar_y_registrar(
    Ridge(),
    nombre="candidato_ridge",
    familia="lineal_regularizado",
    fase="candidato",
    entrenamiento=entrenamiento,
)
resumen(cv)

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run candidato_ridge at: http://localhost:5000/#/experiments/1/runs/406c2ab2a2e04e4698bc2056de992fa2
🧪 View experiment at: http://localhost:5000/#/experiments/1
WAPE train: 0.1326 | WAPE CV: 0.1210 ± 0.0091 | brecha: -0.0116
Costo CV: 0.1865


In [5]:
import importlib, src.pipeline
importlib.reload(src.pipeline)
from src.pipeline import evaluar_y_registrar
from sklearn.ensemble import HistGradientBoostingRegressor
from src.pipeline import SEMILLA

run_id, cv = evaluar_y_registrar(
    HistGradientBoostingRegressor(random_state=SEMILLA),
    nombre="candidato_hist_gradient_boosting",
    familia="boosting",
    fase="candidato",
    entrenamiento=entrenamiento,
)
resumen(cv)

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run candidato_hist_gradient_boosting at: http://localhost:5000/#/experiments/1/runs/d8f31eb057a8467296e9775139950e10
🧪 View experiment at: http://localhost:5000/#/experiments/1
WAPE train: 0.0927 | WAPE CV: 0.0908 ± 0.0048 | brecha: -0.0019
Costo CV: 0.1395


In [6]:
from sklearn.ensemble import RandomForestRegressor

run_id, cv = evaluar_y_registrar(
    RandomForestRegressor(max_samples=0.1, n_jobs=2, random_state=SEMILLA),
    nombre="candidato_random_forest",
    familia="bagging",
    fase="candidato",
    entrenamiento=entrenamiento,
)
resumen(cv)

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run candidato_random_forest at: http://localhost:5000/#/experiments/1/runs/646a228f48814e09946f478e1e646dc8
🧪 View experiment at: http://localhost:5000/#/experiments/1
WAPE train: 0.0787 | WAPE CV: 0.0884 ± 0.0027 | brecha: 0.0097
Costo CV: 0.1396


In [8]:
import importlib, src.pipeline
importlib.reload(src.pipeline)
from src.pipeline import evaluar_y_registrar, SEMILLA

from sklearn.compose import TransformedTargetRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

mlp = TransformedTargetRegressor(
    regressor=MLPRegressor(early_stopping=True, random_state=SEMILLA),
    transformer=StandardScaler(),
)

run_id, cv = evaluar_y_registrar(
    mlp,
    nombre="candidato_mlp",
    familia="red_neuronal",
    fase="candidato",
    entrenamiento=entrenamiento,
)
resumen(cv)

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run candidato_mlp at: http://localhost:5000/#/experiments/1/runs/e91ed69e756d4dc9b2f75b837f7db079
🧪 View experiment at: http://localhost:5000/#/experiments/1
WAPE train: 0.0833 | WAPE CV: 0.0944 ± 0.0063 | brecha: 0.0110
Costo CV: 0.1472


In [9]:
import pandas as pd
from src.datos import cargar_obt

COVARIABLES = ["is_open", "is_promo", "is_state_holiday", "is_school_holiday"]

obt = cargar_obt()[["store_id", "sales_date", "sales"] + COVARIABLES]
obt["sales_date"] = pd.to_datetime(obt["sales_date"])
obt[COVARIABLES] = obt[COVARIABLES].astype(float)

# Calendario completo por tienda: los días sin registro (hueco de 2014) quedan vacíos
fechas = pd.date_range(obt["sales_date"].min(), obt["sales_date"].max(), freq="D")
completo = pd.MultiIndex.from_product([obt["store_id"].unique(), fechas],
                                      names=["store_id", "sales_date"])
series = obt.set_index(["store_id", "sales_date"]).reindex(completo).reset_index()
del obt
gc.collect()

series = series.rename(columns={"store_id": "id", "sales_date": "timestamp", "sales": "target"})
series[COVARIABLES] = series[COVARIABLES].fillna(0)

print("Filas:", len(series), "| Ventas vacías (hueco de 2014):", series["target"].isna().sum())

Filas: 1050330 | Ventas vacías (hueco de 2014): 33121


In [11]:
import time
import numpy as np
from chronos import BaseChronosPipeline
from src.pipeline import CORTES_CV, DIAS_VALIDACION, wape, costo_turnos

chronos = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")

resultados, predicciones = [], []
inicio_total = time.time()

for inicio in pd.to_datetime(CORTES_CV):
    fin = inicio + pd.Timedelta(days=DIAS_VALIDACION)
    contexto = series[series["timestamp"] < inicio]
    futuro = series[(series["timestamp"] >= inicio) & (series["timestamp"] < fin)].drop(columns="target")

    pred = chronos.predict_df(
        contexto, future_df=futuro, prediction_length=DIAS_VALIDACION,
        quantile_levels=[0.5], id_column="id", timestamp_column="timestamp", target="target",
    )
    col = "0.5" if "0.5" in pred.columns else "predictions"
    pred = pred.rename(columns={"id": "store_id", "timestamp": "sales_date", col: "pred"})

    # Se evalúa en las mismas filas que los demás modelos
    bloque = entrenamiento[(entrenamiento["sales_date"] >= inicio) & (entrenamiento["sales_date"] < fin)]
    bloque = bloque[["store_id", "sales_date", "sales"]].merge(
        pred[["store_id", "sales_date", "pred"]], on=["store_id", "sales_date"], how="left")
    bloque["pred"] = bloque["pred"].clip(lower=0)

    resultados.append({"wape": wape(bloque["sales"], bloque["pred"]), "costo": costo_turnos(bloque["sales"], bloque["pred"])})
    predicciones.append(bloque)
    print(f"Pliegue desde {inicio.date()}: WAPE {resultados[-1]['wape']:.4f} | sin predicción: {bloque['pred'].isna().sum()}")

segundos = time.time() - inicio_total
res = pd.DataFrame(resultados)
print(f"\nWAPE CV: {res['wape'].mean():.4f} ± {res['wape'].std(ddof=0):.4f} | Costo CV: {res['costo'].mean():.4f} | {segundos:.0f} s")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--amazon--chronos-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\pc\Desktop\pset_3\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` ca

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Pliegue desde 2015-02-14: WAPE 0.0769 | sin predicción: 0
Pliegue desde 2015-03-28: WAPE 0.0894 | sin predicción: 0
Pliegue desde 2015-05-09: WAPE 0.0871 | sin predicción: 0

WAPE CV: 0.0845 ± 0.0054 | Costo CV: 0.1397 | 1644 s


In [12]:
import os
import tempfile
import mlflow
from src.pipeline import commit_git, hash_datos, SEMILLA, TABLA_ORIGEN

with mlflow.start_run(run_name="candidato_chronos2_desde_corte"):
    mlflow.log_params({
        "modelo": "amazon/chronos-2",
        "modo": "zero-shot",
        "covariables": ",".join(COVARIABLES),
        "cuantil": 0.5,
        "contexto": "toda la historia hasta el corte",
        "semilla": SEMILLA,
        "datos_tabla": TABLA_ORIGEN,
        "datos_fecha_corte": str(entrenamiento["sales_date"].max().date()),
        "datos_hash": hash_datos(series.drop(columns="target"), series["target"]),
    })
    mlflow.log_metrics({
        "wape_cv_media": res["wape"].mean(),
        "wape_cv_std": res["wape"].std(ddof=0),
        "costo_cv_media": res["costo"].mean(),
        "costo_cv_std": res["costo"].std(ddof=0),
        "segundos_inferencia": segundos,
    })
    mlflow.set_tags({
        "fase": "candidato", "familia": "state_of_the_art",
        "git_commit": commit_git(), "evaluacion": "desde_corte_con_ventaja",
    })
    with tempfile.TemporaryDirectory() as carpeta:
        ruta = os.path.join(carpeta, "predicciones_cv.csv")
        pd.concat(predicciones).to_csv(ruta, index=False)
        mlflow.log_artifact(ruta)

print("Registrado")

🏃 View run candidato_chronos2_desde_corte at: http://localhost:5000/#/experiments/1/runs/eb896fad9894403b9365802a854ad9a2
🧪 View experiment at: http://localhost:5000/#/experiments/1
Registrado


In [13]:
resultados_justo, predicciones_justo = [], []
inicio_total = time.time()

for inicio in pd.to_datetime(CORTES_CV):
    fin = inicio + pd.Timedelta(days=DIAS_VALIDACION)
    preds_pliegue = []

    for semana in range(6):
        s = inicio + pd.Timedelta(days=7 * semana)
        fin_semana = s + pd.Timedelta(days=7)
        ultimo_conocido = s - pd.Timedelta(days=42)

        contexto = series[series["timestamp"] <= ultimo_conocido]
        futuro = series[(series["timestamp"] > ultimo_conocido) & (series["timestamp"] < fin_semana)].drop(columns="target")

        pred = chronos.predict_df(
            contexto, future_df=futuro, prediction_length=48,
            quantile_levels=[0.5], id_column="id", timestamp_column="timestamp", target="target",
        )
        col = "0.5" if "0.5" in pred.columns else "predictions"
        pred = pred.rename(columns={"id": "store_id", "timestamp": "sales_date", col: "pred"})
        pred = pred[(pred["sales_date"] >= s) & (pred["sales_date"] < fin_semana)]
        preds_pliegue.append(pred[["store_id", "sales_date", "pred"]])
        print(f"  Pliegue {inicio.date()} | semana {semana + 1}/6 lista | {(time.time() - inicio_total) / 60:.0f} min")

    pred = pd.concat(preds_pliegue)
    bloque = entrenamiento[(entrenamiento["sales_date"] >= inicio) & (entrenamiento["sales_date"] < fin)]
    bloque = bloque[["store_id", "sales_date", "sales"]].merge(pred, on=["store_id", "sales_date"], how="left")
    bloque["pred"] = bloque["pred"].clip(lower=0)

    resultados_justo.append({"wape": wape(bloque["sales"], bloque["pred"]), "costo": costo_turnos(bloque["sales"], bloque["pred"])})
    predicciones_justo.append(bloque)
    print(f"Pliegue desde {inicio.date()}: WAPE {resultados_justo[-1]['wape']:.4f} | sin predicción: {bloque['pred'].isna().sum()}")

segundos_justo = time.time() - inicio_total
res_justo = pd.DataFrame(resultados_justo)
print(f"\nWAPE CV: {res_justo['wape'].mean():.4f} ± {res_justo['wape'].std(ddof=0):.4f} | Costo CV: {res_justo['costo'].mean():.4f} | {segundos_justo:.0f} s")

  Pliegue 2015-02-14 | semana 1/6 lista | 9 min
  Pliegue 2015-02-14 | semana 2/6 lista | 18 min
  Pliegue 2015-02-14 | semana 3/6 lista | 28 min
  Pliegue 2015-02-14 | semana 4/6 lista | 37 min
  Pliegue 2015-02-14 | semana 5/6 lista | 46 min
  Pliegue 2015-02-14 | semana 6/6 lista | 55 min
Pliegue desde 2015-02-14: WAPE 0.0926 | sin predicción: 0
  Pliegue 2015-03-28 | semana 1/6 lista | 64 min
  Pliegue 2015-03-28 | semana 2/6 lista | 74 min
  Pliegue 2015-03-28 | semana 3/6 lista | 83 min
  Pliegue 2015-03-28 | semana 4/6 lista | 92 min
  Pliegue 2015-03-28 | semana 5/6 lista | 102 min
  Pliegue 2015-03-28 | semana 6/6 lista | 111 min
Pliegue desde 2015-03-28: WAPE 0.1005 | sin predicción: 0
  Pliegue 2015-05-09 | semana 1/6 lista | 120 min
  Pliegue 2015-05-09 | semana 2/6 lista | 130 min
  Pliegue 2015-05-09 | semana 3/6 lista | 140 min
  Pliegue 2015-05-09 | semana 4/6 lista | 149 min
  Pliegue 2015-05-09 | semana 5/6 lista | 160 min
  Pliegue 2015-05-09 | semana 6/6 lista | 171

In [14]:
with mlflow.start_run(run_name="candidato_chronos2_igualdad_condiciones"):
    mlflow.log_params({
        "modelo": "amazon/chronos-2",
        "modo": "zero-shot",
        "covariables": ",".join(COVARIABLES),
        "cuantil": 0.5,
        "contexto": "historia hasta 42 dias antes de cada semana pronosticada",
        "semilla": SEMILLA,
        "datos_tabla": TABLA_ORIGEN,
        "datos_fecha_corte": str(entrenamiento["sales_date"].max().date()),
        "datos_hash": hash_datos(series.drop(columns="target"), series["target"]),
    })
    mlflow.log_metrics({
        "wape_cv_media": res_justo["wape"].mean(),
        "wape_cv_std": res_justo["wape"].std(ddof=0),
        "costo_cv_media": res_justo["costo"].mean(),
        "costo_cv_std": res_justo["costo"].std(ddof=0),
        "segundos_inferencia": segundos_justo,
    })
    mlflow.set_tags({
        "fase": "candidato", "familia": "state_of_the_art",
        "git_commit": commit_git(), "evaluacion": "igualdad_condiciones",
    })
    with tempfile.TemporaryDirectory() as carpeta:
        ruta = os.path.join(carpeta, "predicciones_cv.csv")
        pd.concat(predicciones_justo).to_csv(ruta, index=False)
        mlflow.log_artifact(ruta)

print("Registrado")

🏃 View run candidato_chronos2_igualdad_condiciones at: http://localhost:5000/#/experiments/1/runs/f0b1c6276daa46fabb362bd5ceff017e
🧪 View experiment at: http://localhost:5000/#/experiments/1
Registrado
